In [1]:
from pathlib import Path

import pandas as pd


SILVER_FILE = Path(
    "../data/processed/support_tickets_clean.csv"
)

df = pd.read_csv(
    SILVER_FILE,
    parse_dates=[
        "date_of_purchase",
        "first_response_at",
        "resolution_at",
    ],
)

print("Shape:", df.shape)
print("Unique ticket IDs:", df["ticket_id"].nunique())
print("Duplicate ticket IDs:", df["ticket_id"].duplicated().sum())

Shape: (8469, 29)
Unique ticket IDs: 8469
Duplicate ticket IDs: 0


In [2]:
dimension_candidates = {
    "customer": ["customer_key"], 
    "product": ["product_purchased"], 
    "issue": ["ticket_type", "ticket_subject"], 
    "channel": ["ticket_channel"],
    "priority": ["ticket_priority"],
    "status": ["ticket_status"],
    "purchase_date": ["date_of_purchase"],
}

for dimension_name, columns in dimension_candidates.items():
    member_count = (
        df[columns]
        .drop_duplicates()
        .shape[0]
    )

    print(
        f"{dimension_name}: "
        f"{member_count:,} members"
    )

customer: 8,469 members
product: 42 members
issue: 80 members
channel: 4 members
priority: 4 members
status: 3 members
purchase_date: 730 members


In [3]:
customer_profile = (
    df.groupby("customer_key")
    .agg(
        ticket_count=(
            "ticket_id", 
            "count",
        ), 
        age_versions=(
            "customer_age", 
            "nunique",
        ),
        gender_versions=(
            "customer_gender",
            "nunique",
        ), 
        name_versions=(
            "customer_name", 
            "nunique",
        ),
    )
    .reset_index()
)

customer_conflicts = customer_profile[
    (customer_profile["age_versions"] > 1) |
    (customer_profile["gender_versions"] > 1) |
    (customer_profile["name_versions"] > 1)
]

print(
    "Customers with conflicting attributes:",
    len(customer_conflicts),
)

display(customer_conflicts.head(10))

Customers with conflicting attributes: 0


,customer_key,ticket_count,age_versions,gender_versions,name_versions


In [4]:
dimension_key_columns = [
    "customer_key",
    "product_purchased",
    "ticket_type",
    "ticket_subject",
    "ticket_channel",
    "ticket_priority",
    "ticket_status",
]

print(
    df[dimension_key_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

customer_key         0
product_purchased    0
ticket_type          0
ticket_subject       0
ticket_channel       0
ticket_priority      0
ticket_status        0
dtype: int64


In [5]:
issue_profile = (
    df.groupby(
        [
            "ticket_type",
            "ticket_subject",
        ], 
        dropna=False,
    )
    .agg(
        ticket_count=(
            "ticket_id", 
            "count",
        )
    )
    .reset_index()
    .sort_values(
        "ticket_count", 
        ascending=False,
    )
)

print(
    "Unique issue combinations:", 
    len(issue_profile)
)

display(issue_profile.head(20))

Unique issue combinations: 80


,ticket_type,ticket_subject,ticket_count
54,Refund Request,Hardware issue,129
30,Cancellation Request,Refund request,126
11,Billing Inquiry,Product compatibility,123
15,Billing Inquiry,Software bug,122
46,Product Inquiry,Refund request,122
72,Technical Issue,Network problem,122
43,Product Inquiry,Product compatibility,121
55,Refund Request,Installation support,119
75,Technical Issue,Product compatibility,119
49,Refund Request,Battery life,119


In [6]:
print(
    "Minimum purchase date:",
    df["date_of_purchase"].min(),
)

print(
    "Maximum purchase date:",
    df["date_of_purchase"].max(),
)

print(
    "Unique purchase dates:",
    df["date_of_purchase"].nunique(),
)

Minimum purchase date: 2020-01-01 00:00:00
Maximum purchase date: 2021-12-30 00:00:00
Unique purchase dates: 730


In [7]:
purchase_date_coverage = (
    df["date_of_purchase"]
    .dt.normalize()
    .sort_values()
    .drop_duplicates()
)

expected_calendar = pd.date_range(
    start=df["date_of_purchase"].min(),
    end=df["date_of_purchase"].max(),
    freq="D",
)

print(
    "Calendar days in full range:", 
    len(expected_calendar),
)

print(
    "Purchase dates present:", 
    len(purchase_date_coverage)
)

print(
    "Missing calendar dates:", 
    len(expected_calendar)
    - len(purchase_date_coverage),
)

Calendar days in full range: 730
Purchase dates present: 730
Missing calendar dates: 0


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model_utils import (
    attach_dimension_key,
    build_dimension,
)

In [9]:
dim_product_preview = build_dimension(
    source=df, 
    attributes=["product_purchased"],
    key_name="product_key",
)

print("DimProduct shape:", dim_product_preview.shape)
print(
    "Products key unique:", 
    dim_product_preview["product_key"].is_unique, 
)

display(dim_product_preview.head(10))



DimProduct shape: (42, 2)
Products key unique: True


,product_key,product_purchased
0,1,Adobe Photoshop
1,2,Amazon Echo
2,3,Amazon Kindle
3,4,Apple AirPods
4,5,Asus ROG
5,6,Autodesk AutoCAD
6,7,Bose QuietComfort
7,8,Bose SoundLink Speaker
8,9,Canon DSLR Camera
9,10,Canon EOS


In [10]:
dim_issue_preview = build_dimension(
    source=df, 
    attributes=[
        "ticket_type", 
        "ticket_subject"
    ], 
    key_name="issue_key"
)

print("DimIssue shape:", dim_issue_preview.shape)
print(
    "Issue key unique:",
    dim_issue_preview["issue_key"].is_unique,
)

display((dim_issue_preview.head(10)))



DimIssue shape: (80, 3)
Issue key unique: True


,issue_key,ticket_type,ticket_subject
0,1,Billing Inquiry,Account access
1,2,Billing Inquiry,Battery life
2,3,Billing Inquiry,Cancellation request
3,4,Billing Inquiry,Data loss
4,5,Billing Inquiry,Delivery problem
5,6,Billing Inquiry,Display issue
6,7,Billing Inquiry,Hardware issue
7,8,Billing Inquiry,Installation support
8,9,Billing Inquiry,Network problem
9,10,Billing Inquiry,Payment issue


In [ ]:
fact_preview = df.copy()

fact_preview = attach_dimension_key(
    source=fact_preview, 
    dimension=dim_product_preview, 
    attributes=["product_purchased"],
    key_name="product_key"
)

fact_preview = attach_dimension_key(
    source=fact_preview,
    dimension=dim_issue_preview,
    attributes=[
        "ticket_type",
        "ticket_subject"
    ], 
    key_name="issue_key",
)

In [ ]:
print("Fact preview rows:", len(fact_preview))

print(
    "Missing product keys:",
    fact_preview["product_key"].isna().sum(),
)

print(
    "Missing issue keys:",
    fact_preview["issue_key"].isna().sum(),
)

print(
    "Unique ticket IDs:",
    fact_preview["ticket_id"].nunique(),
)

display(
    fact_preview[
        [
            "ticket_id",
            "product_purchased",
            "product_key",
            "ticket_type",
            "ticket_subject",
            "issue_key"
        ]
    ].head(10)
)